# A6 GAN variants — core

Adapted from University of Michigan EECS 498/598, Deep Learning for Computer Vision (Winter 2022), by Justin Johnson and course staff.

**Static validation only; not executed.** Enable the run switches explicitly; generated files go to `core_results/`.

In [ ]:
from pathlib import Path
import os
import sys
import torch
import matplotlib.pyplot as plt
candidates = []
for base in (Path.cwd(), *Path.cwd().parents,
             Path('/content/drive/MyDrive/umich-dlcv-wi2022')):
    candidates.extend((base, base / 'assignments' / 'a6'))
ASSIGNMENT_DIR = next((p.resolve() for p in candidates if (p / 'gan_core.py').is_file()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError('Open from the project/assignment directory, or mount the canonical Colab Drive project first.')
loaded_helpers = sys.modules.get('eecs598')
if loaded_helpers is not None:
    loaded_path = Path(loaded_helpers.__file__).resolve().parent
    if loaded_path != (ASSIGNMENT_DIR / 'eecs598').resolve():
        raise RuntimeError('Restart the kernel before switching assignments: eecs598 is already loaded from another assignment.')
sys.path.insert(0, str(ASSIGNMENT_DIR))
os.chdir(ASSIGNMENT_DIR)
from eecs598.utils import reset_seed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ASSIGNMENT_DIR / 'core_results' / 'generative_adversarial_networks'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_OVERFIT = False
ALLOW_DOWNLOAD = False
reset_seed(0)
plt.rcParams['image.cmap'] = 'gray'
print('Outputs:', OUTPUT_DIR)

In [ ]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from gan_core import (sample_noise, discriminator, generator, discriminator_loss, generator_loss,
    get_optimizer, ls_discriminator_loss, ls_generator_loss, build_dc_classifier, build_dc_generator)
from a6_helper_core import initialize_weights, show_images
BATCH_SIZE, NOISE_DIM = 128, 96
CONFIGS = {
    'fc': (discriminator, generator, discriminator_loss, generator_loss, 10),
    'ls': (discriminator, generator, ls_discriminator_loss, ls_generator_loss, 10),
    'dc': (build_dc_classifier, build_dc_generator, discriminator_loss, generator_loss, 5),
}

## Full GAN, least-squares GAN, and DCGAN training
Use the course losses and optimizer. Real MNIST values are mapped to −1…1; both updates use independent noise samples.

In [ ]:
def run_gan(D, G, loader, d_loss, g_loss, epochs):
    d_optimizer, g_optimizer = get_optimizer(D), get_optimizer(G)
    D.train()
    G.train()
    for epoch in range(epochs):
        for images, _ in loader:
            real = images.to(DEVICE).reshape(-1, 784)
            d_optimizer.zero_grad()
            fake = G(sample_noise(len(real), NOISE_DIM, dtype=real.dtype, device=DEVICE)).detach()
            loss_d = d_loss(D(2 * (real - 0.5)), D(fake))
            loss_d.backward()
            d_optimizer.step()
            g_optimizer.zero_grad()
            fake = G(sample_noise(len(real), NOISE_DIM, dtype=real.dtype, device=DEVICE))
            loss_g = g_loss(D(fake))
            loss_g.backward()
            g_optimizer.step()
        print(f'Epoch {epoch + 1}/{epochs}: D={loss_d.item():.4f}, G={loss_g.item():.4f}')

In [ ]:
if RUN_TRAINING:
    mnist = MNIST(str(ASSIGNMENT_DIR / 'MNIST_data'), train=True, download=ALLOW_DOWNLOAD, transform=ToTensor())
    loader_train = torch.utils.data.DataLoader(mnist, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=0)
    for name, (make_d, make_g, d_loss, g_loss, epochs) in CONFIGS.items():
        reset_seed(0)
        D, G = make_d().to(DEVICE), make_g().to(DEVICE)
        if name == 'dc':
            D.apply(initialize_weights)
            G.apply(initialize_weights)
        run_gan(D, G, loader_train, d_loss, g_loss, epochs)
        torch.save({'discriminator': D.state_dict(), 'generator': G.state_dict()}, OUTPUT_DIR / f'{name}_gan.pt')

## Final generator grids and DCGAN interpolation

In [ ]:
if RUN_EVALUATION:
    reset_seed(0)
    generators = {}
    for name, (_, make_g, _, _, _) in CONFIGS.items():
        checkpoint = OUTPUT_DIR / f'{name}_gan.pt'
        if not checkpoint.is_file():
            raise FileNotFoundError(f'Run training first: {checkpoint}')
        G = make_g().to(DEVICE)
        G.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True)['generator'])
        G.eval()
        generators[name] = G
        with torch.no_grad():
            samples = G(sample_noise(16, NOISE_DIM, device=DEVICE)).cpu()
        show_images(samples)
        plt.savefig(OUTPUT_DIR / f'{name}_gan_results.jpg', bbox_inches='tight')
        plt.show()
    S = 12
    z0 = sample_noise(S, NOISE_DIM, device=DEVICE)
    z1 = sample_noise(S, NOISE_DIM, device=DEVICE)
    weights = torch.linspace(0, 1, S, device=DEVICE).view(S, 1, 1)
    z = (weights * z0 + (1 - weights) * z1).transpose(0, 1).reshape(S * S, NOISE_DIM)
    with torch.no_grad():
        interpolated = generators['dc'](z).cpu()
    show_images(interpolated)
    plt.savefig(OUTPUT_DIR / 'dcgan_interpolation.jpg', bbox_inches='tight')
    plt.show()